# Check options dataset MVP sanity-check

Цель этого notebook: быстро и прагматично проверить, жизнеспособен ли текущий опционный датасет для research MVP по моделям ценообразования опционов.

Проверяем не institutional-grade качество, а достаточность для:
- implied volatility;
- Black–Scholes pricing;
- comparison model price vs market price;
- model error analysis;
- market regime analysis.

Рабочий датасет: семейство `MM` как наиболее однородное и наиболее релевантное для опционов на фьючерс на IMOEX.

In [ ]:
from pathlib import Path
import math

import pandas as pd

In [ ]:
project_root = Path("/Users/maria/Desktop/Code/HSE/COURSEBOOK")
options_path = project_root / "data/imoex_options_mm/imoex_options_mm_daily_history_2024_2026.parquet"
underlying_path = project_root / "data/raw/moex_imoex_2y_candles.parquet"

options = pd.read_parquet(options_path)
underlying = pd.read_parquet(underlying_path)

options["TRADEDATE"] = pd.to_datetime(options["TRADEDATE"], errors="coerce")
options["expiry_date"] = pd.to_datetime(options["expiry_date"], errors="coerce")
options["strike"] = pd.to_numeric(options["strike"], errors="coerce")
options["SETTLEPRICE"] = pd.to_numeric(options["SETTLEPRICE"], errors="coerce")

underlying["TRADEDATE"] = pd.to_datetime(underlying["begin"], errors="coerce").dt.normalize()
underlying["underlying_price"] = pd.to_numeric(underlying["close"], errors="coerce")
underlying_small = underlying[["TRADEDATE", "underlying_price"]].drop_duplicates()

df = options.merge(underlying_small, on="TRADEDATE", how="left")
df["dte_days"] = (df["expiry_date"] - df["TRADEDATE"]).dt.days
df["year"] = df["TRADEDATE"].dt.year
df["option_price"] = df["SETTLEPRICE"]

print(df.shape)
df.head()

## 1. Базовая структура данных

In [ ]:
required_cols = [
    "TRADEDATE",
    "expiry_date",
    "strike",
    "option_type",
    "option_price",
    "underlying_price",
]

structure_summary = pd.DataFrame(
    {
        "column": required_cols,
        "exists": [col in df.columns for col in required_cols],
        "dtype": [str(df[col].dtype) if col in df.columns else None for col in required_cols],
        "non_null_share": [float(df[col].notna().mean()) if col in df.columns else None for col in required_cols],
    }
)

negative_checks = {
    "negative_option_price_rows": int((df["option_price"].dropna() < 0).sum()),
    "negative_strike_rows": int((df["strike"].dropna() < 0).sum()),
    "negative_underlying_rows": int((df["underlying_price"].dropna() < 0).sum()),
}

option_type_values = sorted(df["option_type"].dropna().astype(str).unique().tolist())
valid_option_type_only = set(option_type_values).issubset({"C", "P", "CALL", "PUT", "call", "put"})

structure_summary

In [ ]:
negative_checks, option_type_values, valid_option_type_only

## 2. Missing values

In [ ]:
key_cols = [
    "TRADEDATE",
    "expiry_date",
    "strike",
    "option_type",
    "option_price",
    "underlying_price",
    "OPEN",
    "HIGH",
    "LOW",
    "CLOSE",
    "VALUE",
    "VOLUME",
]

missing_summary = (
    df[key_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
    .to_frame()
)
missing_summary

## 3. Покрытие данных

In [ ]:
coverage = {
    "date_min": df["TRADEDATE"].min().date(),
    "date_max": df["TRADEDATE"].max().date(),
    "unique_trade_days": int(df["TRADEDATE"].nunique()),
    "unique_expirations": int(df["expiry_date"].nunique()),
    "unique_strikes": int(df["strike"].nunique()),
    "rows": len(df),
    "unique_secids": int(df["SECID"].nunique()),
    "rows_by_year": df["year"].value_counts().sort_index().to_dict(),
}

by_date = df.groupby("TRADEDATE").agg(
    options_count=("SECID", "size"),
    strikes_count=("strike", "nunique"),
    expiries_count=("expiry_date", "nunique"),
)

coverage_checks = {
    "share_dates_with_3plus_strikes": float((by_date["strikes_count"] >= 3).mean()),
    "share_dates_with_2plus_expiries": float((by_date["expiries_count"] >= 2).mean()),
}

coverage, coverage_checks

In [ ]:
by_date.describe()

## 4. Cross-section sanity

In [ ]:
sample_dates = (
    pd.Series(sorted(df["TRADEDATE"].dropna().unique()))
    .sample(5, random_state=42)
    .sort_values()
    .tolist()
)

cross_sections = []
for trade_date in sample_dates:
    subset = df.loc[df["TRADEDATE"] == trade_date].copy()
    dte = subset["dte_days"].dropna()
    cross_sections.append(
        {
            "TRADEDATE": pd.Timestamp(trade_date).date(),
            "options_count": int(len(subset)),
            "strikes_count": int(subset["strike"].nunique()),
            "expiries_count": int(subset["expiry_date"].nunique()),
            "dte_min": int(dte.min()) if not dte.empty else None,
            "dte_median": float(dte.median()) if not dte.empty else None,
            "dte_max": int(dte.max()) if not dte.empty else None,
        }
    )

pd.DataFrame(cross_sections)

## 5. Liquidity / garbage check

In [ ]:
garbage_checks = {
    "zero_option_price_share": float((df["option_price"].fillna(0) == 0).mean()),
    "zero_volume_share": float((df["VOLUME"].fillna(0) == 0).mean()),
    "duplicate_secid_tradedate_rows": int(df.duplicated(["SECID", "TRADEDATE"]).sum()),
    "expiry_before_trade_rows": int((df["expiry_date"] < df["TRADEDATE"]).sum()),
    "dte_equal_zero_rows": int((df["dte_days"] == 0).sum()),
}

strike_quantiles = df["strike"].quantile([0.01, 0.05, 0.5, 0.95, 0.99])
price_quantiles = df["option_price"].quantile([0.01, 0.05, 0.5, 0.95, 0.99])

garbage_checks

In [ ]:
strike_quantiles, price_quantiles

## 6. Minimal implied volatility test

In [ ]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_price(spot: float, strike: float, time_to_expiry: float, rate: float, sigma: float, option_type: str) -> float:
    if time_to_expiry <= 0:
        if option_type == "C":
            return max(0.0, spot - strike)
        return max(0.0, strike - spot)
    if sigma <= 0:
        if option_type == "C":
            return max(0.0, spot - strike * math.exp(-rate * time_to_expiry))
        return max(0.0, strike * math.exp(-rate * time_to_expiry) - spot)

    d1 = (
        math.log(spot / strike)
        + (rate + 0.5 * sigma * sigma) * time_to_expiry
    ) / (sigma * math.sqrt(time_to_expiry))
    d2 = d1 - sigma * math.sqrt(time_to_expiry)

    if option_type == "C":
        return spot * norm_cdf(d1) - strike * math.exp(-rate * time_to_expiry) * norm_cdf(d2)
    return strike * math.exp(-rate * time_to_expiry) * norm_cdf(-d2) - spot * norm_cdf(-d1)


def implied_volatility(price: float, spot: float, strike: float, time_to_expiry: float, rate: float, option_type: str) -> float | None:
    if min(price, spot, strike, time_to_expiry) <= 0:
        return None

    if option_type == "C":
        intrinsic = max(0.0, spot - strike * math.exp(-rate * time_to_expiry))
    else:
        intrinsic = max(0.0, strike * math.exp(-rate * time_to_expiry) - spot)

    if price < intrinsic - 1e-8:
        return None

    low_sigma = 1e-4
    high_sigma = 5.0
    low_price = bs_price(spot, strike, time_to_expiry, rate, low_sigma, option_type)
    high_price = bs_price(spot, strike, time_to_expiry, rate, high_sigma, option_type)

    if price < low_price - 1e-8 or price > high_price + 1e-8:
        return None

    left = low_sigma
    right = high_sigma
    for _ in range(100):
        mid = 0.5 * (left + right)
        mid_price = bs_price(spot, strike, time_to_expiry, rate, mid, option_type)
        if abs(mid_price - price) < 1e-6:
            return mid
        if mid_price < price:
            left = mid
        else:
            right = mid
    return 0.5 * (left + right)


In [ ]:
iv_sample = df.loc[
    df["underlying_price"].notna()
    & (df["underlying_price"] > 0)
    & (df["option_price"] > 0)
    & (df["strike"] > 0)
    & (df["dte_days"] > 7)
].copy()

iv_sample = iv_sample.sample(min(300, len(iv_sample)), random_state=42).copy()
iv_sample["time_to_expiry"] = iv_sample["dte_days"] / 365.0
iv_sample["iv_bs"] = iv_sample.apply(
    lambda row: implied_volatility(
        price=float(row["option_price"]),
        spot=float(row["underlying_price"]),
        strike=float(row["strike"]),
        time_to_expiry=float(row["time_to_expiry"]),
        rate=0.15,
        option_type=str(row["option_type"]),
    ),
    axis=1,
)

iv_success_share = float(iv_sample["iv_bs"].notna().mean()) if len(iv_sample) else None
iv_distribution = iv_sample["iv_bs"].dropna().describe()
iv_tail_checks = {
    "iv_success_share": iv_success_share,
    "share_iv_above_3": float((iv_sample["iv_bs"].dropna() > 3.0).mean()) if iv_sample["iv_bs"].notna().any() else None,
    "share_iv_above_5": float((iv_sample["iv_bs"].dropna() > 5.0).mean()) if iv_sample["iv_bs"].notna().any() else None,
}

iv_tail_checks, iv_distribution

## 7. Итоговый verdict

In [ ]:
verdict_lines = [
    "1. Датасет в целом пригоден для research MVP: структура понятная, ключевые поля для опционов есть, strike / dates / option price парсятся корректно.",
    "2. Для option price разумно использовать SETTLEPRICE: он заполнен практически полностью и не содержит отрицательных значений.",
    "3. Cross-section выглядит живым: на торговую дату приходится не 1-2 опциона, а десятки контрактов, много страйков и несколько expiries.",
    "4. Поле underlying_price пока слабое место: в текущей сборке оно покрывает в основном 2024, потому что присоединён только доступный IMOEX candles файл.",
    "5. Поэтому для полного end-to-end pricing/IV исследования на всём диапазоне 2024-2026 нужно ещё догрузить или присоединить underlying prices для 2025-2026.",
    "6. При этом минимальный IV test на пересечении с доступным underlying проходит нормально: значимая доля IV считается успешно, без массовых 500-1000% значений.",
    "7. Практический вывод: датасет жизнеспособен для MVP исследования и уже позволяет переходить к pricing/IV/error analysis хотя бы на покрытом подмножестве; для полноценного 2024-2026 анализа стоит отдельно добрать underlying time series.",
]

print("\n".join(verdict_lines))